# 03 - Predict Real Bulk and Plot Validation

This notebook loads the saved model checkpoint, predicts proportions for `raw_counts.csv`, computes real proportions from `sc_metadata.csv`, reports RMSE/Pearson/R2, and saves validation plots.


In [209]:
from __future__ import annotations

import importlib.util
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Sequence

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "torch": "torch",
}
missing_packages = [name for name, module in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    raise ImportError(
        "Missing required packages: "
        + ", ".join(missing_packages)
        + ". Install the project environment first, for example with `uv sync`."
    )

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root, where pyproject.toml is located.")

REFERENCE_LABEL = "mouse_sub_std"

GRANULARITY = "main" if "main" in REFERENCE_LABEL else "sub"

REFERENCE_DIR = PROJECT_ROOT / "data" / "references" / REFERENCE_LABEL
MODEL_DIR = REFERENCE_DIR / "model"
CHECKPOINT_PATH = MODEL_DIR / "mlp_deconv_checkpoint.pt"
checkpoint_path = CHECKPOINT_PATH

VALIDATION_DATASET = "SNB_EXP_1054"
VALIDATION_SAMPLE_COLUMN = "sample"
VALIDATION_CELLTYPE_COLUMN = "sub_celltype_luee"

# # For PD10 or NG32654
# VALIDATION_BULK_PATH = PROJECT_ROOT / "data" / "test_datasets" / VALIDATION_DATASET / "raw_counts.csv"
# VALIDATION_METADATA_PATH = PROJECT_ROOT / "data" / "test_datasets" / VALIDATION_DATASET /  "sc_metadata.csv"
# PLOTS_DIR =  PROJECT_ROOT / "data" / "test_datasets" / VALIDATION_DATASET /  f"{GRANULARITY}_std"
# VALIDATION_DIR = PROJECT_ROOT / "data" / "test_datasets" / VALIDATION_DATASET /  f"{GRANULARITY}_std"

# For pseudobulks dataset
VALIDATION_BULK_PATH = PROJECT_ROOT / "data" / "test_datasets" / VALIDATION_DATASET / f"{GRANULARITY}_std" / "raw_counts.csv"
VALIDATION_METADATA_PATH = PROJECT_ROOT / "data" / "test_datasets" / VALIDATION_DATASET / f"{GRANULARITY}_std" / "metadata.csv"
PLOTS_DIR =  PROJECT_ROOT / "data" / "test_datasets" / VALIDATION_DATASET / f"{GRANULARITY}_std"
VALIDATION_DIR = PROJECT_ROOT / "data" / "test_datasets" / VALIDATION_DATASET / f"{GRANULARITY}_std"

RANDOM_SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


def require_file(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")


CELLTYPE_COLORS = {
    "neurons": "#D35400",
    "exc_neurons": "#E67E22",
    "inh_neurons": "#F39C12",
    "M/M": "#2E8B57",
    "microglia": "#006D5B",
    "macrophages": "#455A64",
    "hom_mg": "#00A087",
    "inf_mg": "#91D1C2",
    "prol_mg": "#008280",
    "prol_infil_mg": "#4DBBD5",
    "oligodendrocytes": "#5E548E",
    "opc": "#9F86C0",
    "astrocytes": "#BE95C4",
    "endothelial_cells": "#2B4162",
    "epithelial_cells": "#4A6FA5",
    "fibroblasts": "#D6CCC2",
    "b_cells": "#631879",
    "t_cells": "#A20056",
}

CELLTYPE_ORDER = [
    "exc_neurons",
    "inh_neurons",
    "neurons",
    "M/M",
    "hom_mg",
    "inf_mg",
    "prol_mg",
    "prol_infil_mg",
    "microglia",
    "macrophages",
    "oligodendrocytes",
    "opc",
    "astrocytes",
    "endothelial_cells",
    "epithelial_cells",
    "fibroblasts",
    "b_cells",
    "t_cells",
]
DEFAULT_CELLTYPE_COLOR = "#9E9E9E"


def standardize_celltype_label(cell_type: str) -> str:
    key = str(cell_type).strip()
    key = key.replace("-", "_").replace(" ", "_")
    key = "_".join(part for part in key.split("_") if part)
    lower_key = key.lower()
    aliases = {
        "m/m": "M/M",
        "m_m": "M/M",
        "mm": "M/M",
        "b_cell": "b_cells",
        "b_cells": "b_cells",
        "t_cell": "t_cells",
        "t_cells": "t_cells",
        "opc": "opc",
        "opcs": "opc",
        "astrocyte": "astrocytes",
        "astrocytes": "astrocytes",
        "microglia": "microglia",
        "macrophage": "macrophages",
        "macrophages": "macrophages",
        "endothelial_cell": "endothelial_cells",
        "endothelial_cells": "endothelial_cells",
        "epithelial_cell": "epithelial_cells",
        "epithelial_cells": "epithelial_cells",
        "fibroblast": "fibroblasts",
        "fibroblasts": "fibroblasts",
        "neuron": "neurons",
        "neurons": "neurons",
    }
    return aliases.get(lower_key, lower_key)


def order_celltypes(cell_types: Sequence[str]) -> list[str]:
    order_rank = {cell_type: idx for idx, cell_type in enumerate(CELLTYPE_ORDER)}
    return sorted(
        [str(cell_type) for cell_type in cell_types],
        key=lambda cell_type: (
            order_rank.get(standardize_celltype_label(cell_type), len(order_rank)),
            standardize_celltype_label(cell_type),
            str(cell_type),
        ),
    )


def celltype_colors_for(cell_types: Sequence[str]) -> dict[str, str]:
    return {
        str(cell_type): CELLTYPE_COLORS.get(standardize_celltype_label(cell_type), DEFAULT_CELLTYPE_COLOR)
        for cell_type in cell_types
    }


Device: cuda


## Load model utilities


In [210]:
def normalize_bulk_counts(X: np.ndarray, scale_factor: float = 1_000_000.0) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)
    library_sizes = X.sum(axis=1, keepdims=True)
    if np.any(library_sizes <= 0):
        raise ValueError("At least one bulk sample has zero total counts and cannot be normalized.")
    normalized = (X / library_sizes) * scale_factor
    return np.log1p(normalized).astype(np.float32)


def split_train_validation(
    X: np.ndarray,
    y: np.ndarray,
    validation_fraction: float = 0.15,
    seed: int = RANDOM_SEED,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    if X.shape[0] != y.shape[0]:
        raise ValueError("X and y must have the same number of samples.")
    if X.shape[0] < 2:
        raise ValueError("Need at least two pseudobulks to create a validation split.")
    rng = np.random.default_rng(seed)
    indices = rng.permutation(X.shape[0])
    n_validation = int(round(X.shape[0] * validation_fraction))
    n_validation = max(1, min(X.shape[0] - 1, n_validation))
    validation_idx = indices[:n_validation]
    train_idx = indices[n_validation:]
    return X[train_idx], y[train_idx], X[validation_idx], y[validation_idx]


def fit_standardizer(X_train: np.ndarray, eps: float = 1e-6) -> tuple[np.ndarray, np.ndarray]:
    mean = X_train.mean(axis=0).astype(np.float32)
    std = X_train.std(axis=0).astype(np.float32)
    std[std < eps] = 1.0
    return mean, std


def apply_standardizer(X: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    return ((X - mean) / std).astype(np.float32)


In [211]:
def set_reproducible_seed(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class BulkFractionDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.as_tensor(X, dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.float32)

    def __len__(self) -> int:
        return self.X.shape[0]

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.X[index], self.y[index]


class MLPDeconvRegressor(nn.Module):
    def __init__(
        self,
        input_dim: int,
        output_dim: int,
        hidden_sizes: Sequence[int] = (512, 256, 128),
        dropout: float = 0.2,
    ):
        super().__init__()
        layers = []
        previous_dim = input_dim
        for hidden_dim in hidden_sizes:
            layers.append(nn.Linear(previous_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            previous_dim = hidden_dim
        layers.append(nn.Linear(previous_dim, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        logits = self.network(X)
        return torch.softmax(logits, dim=1)


@dataclass
class TrainingConfig:
    hidden_sizes: tuple[int, ...] = (2048, 1024, 512)
    dropout: float = 0.0
    batch_size: int = 256
    learning_rate: float = 0.000213
    weight_decay: float = 6.38e-06
    max_steps: int = 5000
    validation_every: int = 100
    patience: int = 15
    min_delta: float = 1e-5
    loss_name: str = "MSELoss"


In [212]:
@torch.no_grad()
def predict_array(model: nn.Module, X: np.ndarray, batch_size: int = 256, device: torch.device = DEVICE) -> np.ndarray:
    model.eval()
    tensor_dataset = torch.as_tensor(X, dtype=torch.float32)
    loader = DataLoader(tensor_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    predictions = []
    for X_batch in loader:
        X_batch = X_batch.to(device)
        predictions.append(model(X_batch).cpu().numpy())
    return np.vstack(predictions).astype(np.float32)


def rmse_score(y_true: np.ndarray, y_pred: np.ndarray, axis=None) -> np.ndarray:
    return np.sqrt(np.mean(np.square(y_pred - y_true), axis=axis))


def pearson_score(y_true: np.ndarray, y_pred: np.ndarray, axis=None, eps: float = 1e-12) -> np.ndarray:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    if axis is None:
        y_true = y_true.ravel()
        y_pred = y_pred.ravel()
        true_std = y_true.std()
        pred_std = y_pred.std()
        if true_std < eps or pred_std < eps:
            return np.nan
        return float(np.mean((y_true - y_true.mean()) * (y_pred - y_pred.mean())) / (true_std * pred_std))

    true_mean = np.mean(y_true, axis=axis, keepdims=True)
    pred_mean = np.mean(y_pred, axis=axis, keepdims=True)
    centered_true = y_true - true_mean
    centered_pred = y_pred - pred_mean
    numerator = np.mean(centered_true * centered_pred, axis=axis)
    denominator = np.std(y_true, axis=axis) * np.std(y_pred, axis=axis)
    return np.divide(numerator, denominator, out=np.full_like(numerator, np.nan, dtype=np.float64), where=denominator >= eps)


def r2_score(y_true: np.ndarray, y_pred: np.ndarray, axis=None, eps: float = 1e-12) -> np.ndarray:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    ss_res = np.sum(np.square(y_true - y_pred), axis=axis)
    ss_tot = np.sum(np.square(y_true - np.mean(y_true, axis=axis, keepdims=True)), axis=axis)
    return np.divide(1 - (ss_res / ss_tot), 1.0, out=np.full_like(ss_res, np.nan, dtype=np.float64), where=ss_tot >= eps)


def deconvolution_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    cell_types: Sequence[str],
    sample_ids: Sequence[str] | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    if y_true.shape != y_pred.shape:
        raise ValueError(f"Shape mismatch: y_true={y_true.shape}, y_pred={y_pred.shape}")

    if sample_ids is None:
        sample_ids = [f"sample_{idx:05d}" for idx in range(y_true.shape[0])]

    per_celltype = pd.DataFrame(
        {
            "celltype": list(cell_types),
            "rmse": rmse_score(y_true, y_pred, axis=0),
            "pearson": pearson_score(y_true, y_pred, axis=0),
            "r2": r2_score(y_true, y_pred, axis=0),
        }
    ).sort_values("rmse", ascending=True).reset_index(drop=True)

    per_sample = pd.DataFrame(
        {
            "sample_id": list(sample_ids),
            "rmse": rmse_score(y_true, y_pred, axis=1),
            "pearson": pearson_score(y_true, y_pred, axis=1),
            "r2": r2_score(y_true, y_pred, axis=1),
        }
    ).sort_values("rmse", ascending=True).reset_index(drop=True)

    global_metrics = pd.Series(
        {
            "rmse": float(rmse_score(y_true, y_pred)),
            "pearson": float(pearson_score(y_true, y_pred)),
            "r2": float(r2_score(y_true, y_pred)),
        },
        name="metrics",
    )
    return per_celltype, per_sample, global_metrics


def select_example_samples(per_sample_metrics: pd.DataFrame, n_examples: int = 8) -> list[str]:
    metrics = per_sample_metrics.sort_values("rmse", ascending=True).reset_index(drop=True)
    if len(metrics) <= n_examples:
        return metrics["sample_id"].astype(str).tolist()

    n_best = n_examples // 2
    n_worst = n_examples - n_best
    selected = pd.concat([metrics.head(n_best), metrics.tail(n_worst)], axis=0)
    return selected["sample_id"].astype(str).tolist()


def plot_real_vs_predicted_stacked_bars(
    y_true_df: pd.DataFrame,
    y_pred_df: pd.DataFrame,
    sample_ids: Sequence[str],
    celltype_colors: dict[str, str],
    output_path: str | Path,
    title: str,
) -> Path:
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    sample_ids = [str(sample_id) for sample_id in sample_ids]

    # This controls the stacking order.
    # First item is plotted at the bottom of the stacked bar.
    cell_types = list(y_true_df.columns)[::-1]

    # This controls the legend order.
    # Reversing here makes the legend match the visual top-to-bottom stack.
    legend_cell_types = cell_types[::-1]

    x_positions = np.arange(len(sample_ids), dtype=float)
    width = 0.32
    pair_gap = 0.06

    real_positions = x_positions - (width + pair_gap) / 2
    pred_positions = x_positions + (width + pair_gap) / 2

    real_bottom = np.zeros(len(sample_ids), dtype=float)
    pred_bottom = np.zeros(len(sample_ids), dtype=float)

    fig_width = max(10, len(sample_ids) * 1.25)
    fig, ax = plt.subplots(figsize=(fig_width, 5.5))

    for cell_type in cell_types:
        color = celltype_colors.get(cell_type, DEFAULT_CELLTYPE_COLOR)

        real_values = y_true_df.loc[sample_ids, cell_type].to_numpy(dtype=float)
        pred_values = y_pred_df.loc[sample_ids, cell_type].to_numpy(dtype=float)

        ax.bar(
            real_positions,
            real_values,
            width=width,
            bottom=real_bottom,
            color=color,
            edgecolor="white",
            linewidth=0.4,
        )

        ax.bar(
            pred_positions,
            pred_values,
            width=width,
            bottom=pred_bottom,
            color=color,
            edgecolor="white",
            linewidth=0.4,
        )

        real_bottom += real_values
        pred_bottom += pred_values

    ax.set_title(title)
    ax.set_ylabel("Proportion")
    ax.set_ylim(0, 1.05)

    ax.set_xticks(x_positions)
    ax.set_xticklabels(sample_ids, rotation=45, ha="right")

    fig.text(
        0.5,
        0.01,
        "left bar: real    right bar: predicted",
        ha="center",
        va="bottom",
        fontsize=8,
    )

    handles = [
        plt.Rectangle(
            (0, 0),
            1,
            1,
            color=celltype_colors.get(ct, DEFAULT_CELLTYPE_COLOR),
        )
        for ct in legend_cell_types
    ]

    ax.legend(
        handles,
        legend_cell_types,
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False,
        title="Cell type",
    )

    fig.tight_layout(rect=[0, 0.04, 1, 1])
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved stacked real-vs-predicted plot to {output_path}")

    return output_path

In [213]:
def torch_load_checkpoint(path: str | Path, device: torch.device = DEVICE) -> dict:
    path = Path(path)
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def load_trained_model(checkpoint_path: str | Path, device: torch.device = DEVICE) -> tuple[MLPDeconvRegressor, dict]:
    checkpoint = torch_load_checkpoint(checkpoint_path, device=device)
    model_config = checkpoint["model_config"]
    model = MLPDeconvRegressor(
        input_dim=len(checkpoint["genes"]),
        output_dim=len(checkpoint["cell_types"]),
        hidden_sizes=tuple(model_config["hidden_sizes"]),
        dropout=float(model_config["dropout"]),
    ).to(device)
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    return model, checkpoint


def read_bulk_count_csv(file_path: str | Path, genes_are_rows: bool = True) -> pd.DataFrame:
    path = Path(file_path)
    require_file(path, "bulk count matrix")
    bulk_df = pd.read_csv(path, index_col=0)
    if genes_are_rows:
        bulk_df = bulk_df.T
    bulk_df.index = bulk_df.index.astype(str)
    bulk_df.columns = bulk_df.columns.astype(str)
    return bulk_df


def align_bulk_dataframe(
    bulk_df: pd.DataFrame,
    training_genes: Sequence[str],
    genes_are_rows: bool = False,
    fill_missing_genes: bool = False,
) -> pd.DataFrame:
    if genes_are_rows:
        bulk_df = bulk_df.T
    bulk_df = bulk_df.copy()
    bulk_df.index = bulk_df.index.astype(str)
    bulk_df.columns = bulk_df.columns.astype(str)
    if not bulk_df.columns.is_unique:
        raise ValueError("Bulk matrix has duplicated gene columns. Please collapse duplicates before prediction.")

    training_genes = list(training_genes)
    training_gene_set = set(training_genes)
    extra_bulk_genes = [gene for gene in bulk_df.columns if gene not in training_gene_set]
    if extra_bulk_genes:
        print(f"Dropping {len(extra_bulk_genes):,} bulk genes that were not used by the reference/model.")
        bulk_df = bulk_df.loc[:, [gene for gene in bulk_df.columns if gene in training_gene_set]]

    missing_genes = [gene for gene in training_genes if gene not in bulk_df.columns]
    if missing_genes and not fill_missing_genes:
        preview = ", ".join(missing_genes[:10])
        raise ValueError(
            f"Bulk matrix is missing {len(missing_genes):,} training genes. First missing genes: {preview}"
        )
    if missing_genes:
        print(f"Warning: filling {len(missing_genes):,} missing training genes with zeros.")

    return bulk_df.reindex(columns=training_genes, fill_value=0.0)


def predict_bulk_dataframe(
    bulk_df: pd.DataFrame,
    checkpoint_path: str | Path,
    genes_are_rows: bool = False,
    fill_missing_genes: bool = False,
    device: torch.device = DEVICE,
) -> pd.DataFrame:
    trained_model, checkpoint = load_trained_model(checkpoint_path, device=device)
    aligned_bulk = align_bulk_dataframe(
        bulk_df,
        training_genes=checkpoint["genes"],
        genes_are_rows=genes_are_rows,
        fill_missing_genes=fill_missing_genes,
    )
    X_counts = aligned_bulk.to_numpy(dtype=np.float32)
    X_normalized = normalize_bulk_counts(X_counts, scale_factor=float(checkpoint["normalization"]["scale_factor"]))
    X_scaled = apply_standardizer(
        X_normalized,
        np.asarray(checkpoint["gene_mean"], dtype=np.float32),
        np.asarray(checkpoint["gene_std"], dtype=np.float32),
    )
    predictions = predict_array(trained_model, X_scaled, device=device)
    return pd.DataFrame(predictions, index=aligned_bulk.index, columns=checkpoint["cell_types"])


In [214]:
checkpoint_preview = torch_load_checkpoint(CHECKPOINT_PATH, device=DEVICE)
CELL_TYPES = order_celltypes(checkpoint_preview["cell_types"])
CELLTYPE_PLOT_COLORS = celltype_colors_for(CELL_TYPES)
display(pd.DataFrame({"celltype": CELL_TYPES, "color": [CELLTYPE_PLOT_COLORS[ct] for ct in CELL_TYPES]}))


,celltype,color
0,exc_neurons,#E67E22
1,inh_neurons,#F39C12
2,hom_mg,#00A087
3,inf_mg,#91D1C2
4,prol_mg,#008280
5,macrophages,#455A64
6,oligodendrocytes,#5E548E
7,opc,#9F86C0
8,astrocytes,#BE95C4
9,endothelial_cells,#2B4162


## Predict and validate real bulk samples

In [215]:
def load_true_proportions_from_metadata(
    metadata_path: str | Path,
    sample_column: str = VALIDATION_SAMPLE_COLUMN,
    celltype_column: str = VALIDATION_CELLTYPE_COLUMN,
    cell_types: Sequence[str] | None = None,
) -> pd.DataFrame:
    path = Path(metadata_path)
    require_file(path, "validation single-cell metadata")
    metadata = pd.read_csv(path, index_col=0)
    if sample_column not in metadata.columns or celltype_column not in metadata.columns:
        metadata = pd.read_csv(path)
    missing = {sample_column, celltype_column}.difference(metadata.columns)
    if missing:
        raise ValueError(f"Validation metadata is missing columns: {sorted(missing)}")

    sample_labels = metadata[sample_column].astype(str)
    celltype_labels = metadata[celltype_column].astype(str)

    if cell_types is not None:
        model_celltype_by_style_key = {standardize_celltype_label(ct): str(ct) for ct in cell_types}
        celltype_labels = celltype_labels.map(
            lambda label: model_celltype_by_style_key.get(standardize_celltype_label(label), str(label))
        )

    counts = pd.crosstab(sample_labels, celltype_labels)
    proportions = counts.div(counts.sum(axis=1), axis=0)
    proportions.index.name = "sample_id"

    if cell_types is not None:
        cell_types = list(cell_types)
        missing_from_validation = [ct for ct in cell_types if ct not in proportions.columns]
        extra_validation_types = sorted(set(proportions.columns).difference(cell_types))
        if missing_from_validation:
            print(f"Validation metadata has no cells for these training cell types: {missing_from_validation}")
        if extra_validation_types:
            print(f"Validation metadata has extra cell types not used by the model: {extra_validation_types}")
        proportions = proportions.reindex(columns=cell_types, fill_value=0.0)

    return proportions.astype(np.float32)


In [216]:
FILL_MISSING_VALIDATION_GENES = False

validation_bulk = read_bulk_count_csv(VALIDATION_BULK_PATH, genes_are_rows=True)
validation_true = load_true_proportions_from_metadata(
    VALIDATION_METADATA_PATH,
    sample_column=VALIDATION_SAMPLE_COLUMN,
    celltype_column=VALIDATION_CELLTYPE_COLUMN,
    cell_types=CELL_TYPES,
)

common_samples = validation_bulk.index.intersection(validation_true.index)
if len(common_samples) == 0:
    raise ValueError("No sample IDs overlap between raw_counts.csv columns and sc_metadata.csv sample labels.")
if len(common_samples) != validation_bulk.shape[0] or len(common_samples) != validation_true.shape[0]:
    print(f"Using {len(common_samples)} overlapping validation samples.")

validation_bulk = validation_bulk.loc[common_samples]
validation_true = validation_true.loc[common_samples]
validation_predictions = predict_bulk_dataframe(
    validation_bulk,
    checkpoint_path=checkpoint_path,
    fill_missing_genes=FILL_MISSING_VALIDATION_GENES,
    device=DEVICE,
)
validation_predictions = validation_predictions.loc[validation_true.index, CELL_TYPES]

validation_metrics_by_celltype, validation_metrics_by_sample, validation_global_metrics = deconvolution_metrics(
    validation_true.to_numpy(dtype=np.float32),
    validation_predictions.to_numpy(dtype=np.float32),
    CELL_TYPES,
    sample_ids=validation_true.index.tolist(),
)

VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
validation_true.to_csv(VALIDATION_DIR / "validation_true_proportions.csv", index_label="sample_id")
validation_predictions.to_csv(VALIDATION_DIR / "validation_predicted_proportions.csv", index_label="sample_id")
validation_metrics_by_celltype.to_csv(VALIDATION_DIR / "validation_metrics_by_celltype.csv", index=False)
validation_metrics_by_sample.to_csv(VALIDATION_DIR / "validation_metrics_by_sample.csv", index=False)
validation_global_metrics.to_frame().to_csv(VALIDATION_DIR / "validation_metrics_global.csv")

validation_plot_path = plot_real_vs_predicted_stacked_bars(
    validation_true,
    validation_predictions,
    sample_ids=validation_true.index.tolist()[:8],
    celltype_colors=CELLTYPE_PLOT_COLORS,
    output_path=VALIDATION_DIR / "validation_real_vs_predicted_stacked.png",
    title="External validation: real vs predicted proportions",
)

display(validation_global_metrics)
display(validation_metrics_by_celltype)
display(validation_metrics_by_sample)
validation_predictions


Validation metadata has no cells for these training cell types: ['opc', 'epithelial_cells']
Dropping 19,402 bulk genes that were not used by the reference/model.


/tmp/ipykernel_910/2328095196.py:43: RuntimeWarning: divide by zero encountered in divide
  return np.divide(1 - (ss_res / ss_tot), 1.0, out=np.full_like(ss_res, np.nan, dtype=np.float64), where=ss_tot >= eps)


Saved stacked real-vs-predicted plot to /mnt/custom-file-systems/efs/fs-0fb4188b34a49ecf4_fsap-0f622ffecc95fdbba/MLPDeconv/data/test_datasets/SNB_EXP_1054/sub_std/validation_real_vs_predicted_stacked.png


rmse       0.109083
pearson    0.658396
r2        -1.415228
Name: metrics, dtype: float64

,celltype,rmse,pearson,r2
0,epithelial_cells,0.001365,NaN,NaN
1,opc,0.001733,NaN,NaN
2,b_cells,0.007577,0.924137,-2.220964
3,t_cells,0.008113,0.925380,-2.875526
4,macrophages,0.009659,0.483527,-8.965750
5,prol_mg,0.021003,-0.541046,-3.431563
6,hom_mg,0.030613,0.832163,-3.990743
7,inf_mg,0.036906,-0.558327,-35.770266
8,oligodendrocytes,0.064191,0.799484,-9.039832
9,endothelial_cells,0.088581,0.955641,-12.452905


,sample_id,rmse,pearson,r2
0,ID14_cNTG_TG_S4,0.096310,0.694643,-0.740398
1,ID01_Veh_TG_S1,0.100171,0.708995,-1.107022
2,ID04_cNTG_TG_S3,0.111377,0.599341,-1.515652
3,ID11_Veh_TG_S2,0.126024,0.645852,-2.390699


,exc_neurons,inh_neurons,hom_mg,inf_mg,prol_mg,macrophages,oligodendrocytes,opc,astrocytes,endothelial_cells,epithelial_cells,fibroblasts,b_cells,t_cells
ID01_Veh_TG_S1,0.025654,0.354091,0.007441,0.000617,0.003094,0.003785,0.116692,0.001823,0.038696,0.004550,0.001456,0.436963,0.003480,0.001657
ID04_cNTG_TG_S3,0.031787,0.288406,0.002466,0.001080,0.000653,0.001876,0.112226,0.001649,0.054917,0.010760,0.001721,0.488905,0.002326,0.001228
ID11_Veh_TG_S2,0.002777,0.467644,0.004798,0.000944,0.001539,0.002478,0.031325,0.001333,0.026478,0.004245,0.001025,0.452110,0.001393,0.001912
ID14_cNTG_TG_S4,0.046243,0.081015,0.001621,0.001612,0.001207,0.002300,0.222484,0.002047,0.112350,0.033158,0.001147,0.491289,0.001813,0.001715
